# VisionInspect — 04 Domain Routing & YOLO26n Classifier

A lightweight visual domain router for selecting the appropriate inspection specialist.

Domains: DAGM, KolektorSDD2, Magnetic Tile, and MVTec AD.


In [11]:
from pathlib import Path
import random, shutil, json, time
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

DATA_ROOT = Path('datasets')
ROUTER_ROOT = DATA_ROOT / 'domain_router'

SOURCE_ROOTS = {
    'DAGM': DATA_ROOT / 'yolo_segmentation' / 'DAGM',
    'KolektorSDD2': DATA_ROOT / 'yolo_segmentation' / 'KolektorSDD2',
    'Magnetic_Tile': DATA_ROOT / 'yolo_segmentation' / 'Magnetic_Tile',
    'MVTec_AD': DATA_ROOT / 'MVTec_AD',
}

DOMAINS = ['DAGM', 'KolektorSDD2', 'Magnetic_Tile', 'MVTec_AD']
DOMAIN_TO_ID = {name: i for i, name in enumerate(DOMAINS)}
ID_TO_DOMAIN = {i: name for name, i in DOMAIN_TO_ID.items()}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

MODEL_NAME = 'yolo26n-cls.pt'
IMAGE_SIZE = 224
EPOCHS = 50
BATCH_SIZE = 32
WORKERS = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Router:', ROUTER_ROOT)
print('Classes:', DOMAIN_TO_ID)
print('Device:', DEVICE)


Router: datasets/domain_router
Classes: {'DAGM': 0, 'KolektorSDD2': 1, 'Magnetic_Tile': 2, 'MVTec_AD': 3}
Device: cuda


## 1. Check source datasets


In [2]:
for domain, path in SOURCE_ROOTS.items():
    print(('✓' if path.exists() else '✗'), f'{domain:<20}', path)


✓ DAGM                 datasets/yolo_segmentation/DAGM
✓ KolektorSDD2         datasets/yolo_segmentation/KolektorSDD2
✓ Magnetic_Tile        datasets/yolo_segmentation/Magnetic_Tile
✓ MVTec_AD             datasets/MVTec_AD


## 2. Discover images


In [3]:
def find_images(root, exclude_masks=True):
    root = Path(root)

    if not root.exists():
        return []

    images = []

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        # Check extension
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        # Exclude mask/label files
        if exclude_masks:

            filename = path.stem.lower()
            parts = {
                part.lower()
                for part in path.parts
            }

            # Filename-based mask detection
            mask_patterns = (
                "_mask",
                "-mask",
                "_gt",
                "-gt",
                "_label",
                "-label",
            )

            if filename.endswith(mask_patterns):
                continue

            # Directory-based label detection
            label_directories = {
                "mask",
                "masks",
                "label",
                "labels",
                "ground_truth",
                "ground-truth",
                "gt",
            }

            if parts.intersection(label_directories):
                continue

        images.append(path)

    return sorted(images)


def find_prepared_images(root):
    images = []
    root = Path(root)

    for split in ['train', 'val', 'test']:
        directory = root / 'images' / split

        if directory.exists():
            images.extend(find_images(directory))

    return images


In [4]:
def find_prepared_images(root):
    """
    Find images from an already prepared YOLO segmentation dataset.

    Expected:
        root/
            images/
                train/
                val/
                test/
    """
    root = Path(root)
    images = []

    for split in ["train", "val", "test"]:
        directory = root / "images" / split

        if directory.exists():
            images.extend(
                find_images(
                    directory,
                    exclude_masks=True
                )
            )

    return sorted(images)


def find_mvtec_images(root):
    """
    Find actual MVTec AD sample images while excluding
    ground-truth/mask images.
    """
    return find_images(
        root,
        exclude_masks=True
    )

In [5]:
def find_suspicious_images(root):
    suspicious = []

    for path in find_images(root, exclude_masks=False):

        filename = path.stem.lower()

        if filename.endswith(
            ("_mask", "-mask", "_gt", "-gt", "_label", "-label")
        ):
            suspicious.append(path)

    return suspicious


mvtec_suspicious = find_suspicious_images(
    SOURCE_ROOTS["MVTec_AD"]
)

print(
    f"Potential mask files found: "
    f"{len(mvtec_suspicious)}"
)

for path in mvtec_suspicious[:20]:
    print(path)

Potential mask files found: 1258
datasets/MVTec_AD/bottle/ground_truth/broken_large/000_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/001_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/002_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/003_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/004_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/005_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/006_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/007_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/008_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/009_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/010_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/011_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/012_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/013_mask.png
datasets/MVTec_AD/bottle/ground_truth/broken_large/014_mask.png
dataset

## 3. Build domain-labelled inventory


In [6]:
records = []

for domain in ['DAGM', 'KolektorSDD2', 'Magnetic_Tile']:
    for path in find_prepared_images(SOURCE_ROOTS[domain]):
        records.append({'domain': domain, 'source_path': str(path)})

for path in find_images(SOURCE_ROOTS['MVTec_AD']):
    records.append({'domain': 'MVTec_AD', 'source_path': str(path)})

inventory = pd.DataFrame(records)
if inventory.empty:
    raise RuntimeError('No source images found. Check SOURCE_ROOTS.')

print(f'Total images: {len(inventory):,}')
display(inventory.groupby('domain').size().rename('images').to_frame())


Total images: 13,820


,images
domain,
DAGM,3786
KolektorSDD2,3336
MVTec_AD,5354
Magnetic_Tile,1344


## 4. Stratified 70/15/15 router split


In [7]:
train_df, temp_df = train_test_split(
    inventory, test_size=0.30, stratify=inventory['domain'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['domain'], random_state=SEED
)
train_df = train_df.assign(split='train')
val_df = val_df.assign(split='val')
test_df = test_df.assign(split='test')
router_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
display(router_df.groupby(['split', 'domain']).size().unstack(fill_value=0))


domain,DAGM,KolektorSDD2,MVTec_AD,Magnetic_Tile
split,,,,
test,568,501,803,201
train,2650,2335,3748,941
val,568,500,803,202


## 5. Build YOLO classification directory


In [8]:
def build_router_dataset(dataframe, output_root, overwrite=False):
    output_root = Path(output_root)
    if overwrite and output_root.exists():
        shutil.rmtree(output_root)
    for split in ['train', 'val', 'test']:
        for domain in DOMAINS:
            (output_root / split / domain).mkdir(parents=True, exist_ok=True)
    manifest = []
    for index, row in dataframe.reset_index(drop=True).iterrows():
        source = Path(row['source_path'])
        destination = output_root / row['split'] / row['domain'] / f'{index:07d}_{source.name}'
        if not destination.exists():
            shutil.copy2(source, destination)
        manifest.append({
            'index': index, 'split': row['split'], 'domain': row['domain'],
            'source_path': str(source), 'router_path': str(destination)
        })
    return pd.DataFrame(manifest)

manifest = build_router_dataset(router_df, ROUTER_ROOT, overwrite=False)
manifest.to_csv(ROUTER_ROOT / 'manifest.csv', index=False)
print(ROUTER_ROOT.resolve())


/home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router


## 6. Create dataset metadata and validate distribution


In [9]:
router_yaml = {
    'path': str(ROUTER_ROOT), 'train': 'train', 'val': 'val', 'test': 'test',
    'nc': len(DOMAINS), 'names': DOMAINS
}
yaml_path = ROUTER_ROOT / 'dataset.yaml'
yaml_path.write_text(yaml.safe_dump(router_yaml, sort_keys=False), encoding='utf-8')

def count_router_images(root):
    rows = []
    for split in ['train', 'val', 'test']:
        for domain in DOMAINS:
            rows.append({'split': split, 'domain': domain,
                         'images': len(find_images(Path(root) / split / domain))})
    return pd.DataFrame(rows)

router_counts = count_router_images(ROUTER_ROOT)
display(router_counts.pivot(index='domain', columns='split', values='images').fillna(0).astype(int))

train_counts = router_counts[router_counts['split'] == 'train']
print('Imbalance ratio:', train_counts['images'].max() / train_counts['images'].min())


split,test,train,val
domain,,,
DAGM,568,2650,568
KolektorSDD2,501,2335,500
MVTec_AD,803,3748,803
Magnetic_Tile,201,941,202


Imbalance ratio: 3.9829968119022316


## 7. Train YOLO26n domain classifier


In [12]:
model = YOLO(MODEL_NAME)
start_time = time.perf_counter()

model.train(
    data=str(ROUTER_ROOT),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    pretrained=True,
    patience=15,
    project='runs/classify',
    name='domain_router',
    exist_ok=True,
    plots=True,
)

training_time = time.perf_counter() - start_time
print(f'Training time: {training_time / 60:.2f} minutes')
print('Best model:', model.trainer.best)


New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/domain_router, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixu

## 8. Validation


In [13]:
val_results = model.val(
    data=str(ROUTER_ROOT), split='val', imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, device=DEVICE
)
print('Top-1:', getattr(val_results, 'top1', None))
print('Top-5:', getattr(val_results, 'top5', None))


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
YOLO26n-cls summary (fused): 47 layers, 1,531,148 parameters, 0 gradients, 3.2 GFLOPs
train: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router/train... found 9674 images in 4 classes ✅ 
val: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router/val... found 2073 images in 4 classes ✅ 
test: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router/test... found 2073 images in 4 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6171.3±1836.1 MB/s, size: 182.6 KB)
val: Scanning /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router/val... 2073 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2073/2073 966.1Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 65/65 13.4it/s 4.8s0.4s
                   all          1          1
Speed: 0.1ms preprocess, 0.2ms inferen

## 9. Test routing metrics


In [14]:
test_rows = []
for true_domain in DOMAINS:
    for image_path in find_images(ROUTER_ROOT / 'test' / true_domain):
        result = model.predict(source=str(image_path), imgsz=IMAGE_SIZE,
                               device=DEVICE, verbose=False)[0]
        predicted_id = int(result.probs.top1)
        confidence = float(result.probs.top1conf)
        predicted_domain = ID_TO_DOMAIN[predicted_id]
        test_rows.append({
            'image': str(image_path), 'true_domain': true_domain,
            'pred_domain': predicted_domain, 'confidence': confidence,
            'correct': predicted_domain == true_domain
        })

test_predictions = pd.DataFrame(test_rows)
y_true = test_predictions['true_domain']
y_pred = test_predictions['pred_domain']

print(classification_report(y_true, y_pred, labels=DOMAINS,
      target_names=DOMAINS, digits=4, zero_division=0))

routing_accuracy = float(test_predictions['correct'].mean())
wrong_specialist_rate = 1.0 - routing_accuracy
print(f'Routing accuracy: {routing_accuracy:.4%}')
print(f'Wrong-specialist rate: {wrong_specialist_rate:.4%}')


               precision    recall  f1-score   support

         DAGM     1.0000    1.0000    1.0000       568
 KolektorSDD2     1.0000    1.0000    1.0000       501
Magnetic_Tile     0.0000    0.0000    0.0000       201
     MVTec_AD     0.0000    0.0000    0.0000       803

     accuracy                         0.5157      2073
    macro avg     0.5000    0.5000    0.5000      2073
 weighted avg     0.5157    0.5157    0.5157      2073

Routing accuracy: 51.5678%
Wrong-specialist rate: 48.4322%


## 10. Confusion matrix


In [16]:
cm = confusion_matrix(y_true, y_pred, labels=DOMAINS)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=DOMAINS).plot(ax=ax, xticks_rotation=30)
ax.set_title('VisionInspect Domain Router — Test')
plt.tight_layout()
display(plt.show())


<Figure size 800x700 with 2 Axes>

None

## 11. Confidence-based rejection analysis


In [17]:
threshold_rows = []
for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    accepted = test_predictions['confidence'] >= threshold
    threshold_rows.append({
        'threshold': threshold,
        'accepted_accuracy': test_predictions.loc[accepted, 'correct'].mean() if accepted.any() else None,
        'rejection_rate': 1.0 - accepted.mean(),
        'accepted_samples': int(accepted.sum()),
    })
display(pd.DataFrame(threshold_rows))


,threshold,accepted_accuracy,rejection_rate,accepted_samples
0,0.50,0.515678,0.0,2073
1,0.60,0.515678,0.0,2073
2,0.70,0.515678,0.0,2073
3,0.80,0.515678,0.0,2073
4,0.90,0.515678,0.0,2073
5,0.95,0.515678,0.0,2073


## 12. Router latency


In [18]:
sample_images = test_predictions['image'].head(20).tolist()
for path in sample_images:
    model.predict(source=path, imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)

start = time.perf_counter()
for path in sample_images:
    model.predict(source=path, imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)
latency_ms = (time.perf_counter() - start) / len(sample_images) * 1000
print(f'Router latency: {latency_ms:.2f} ms/image')


Router latency: 5.97 ms/image


## 13. Export to ONNX


In [19]:
onnx_path = Path(model.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True))
print('ONNX model:', onnx_path)
print(f'Size: {onnx_path.stat().st_size / (1024 ** 2):.2f} MB')


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CPU (AMD Ryzen AI 9 365 w/ Radeon 880M)

PyTorch: starting from '/home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/domain_router/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 4) (3.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/19.1 MB ? eta -:--:--
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/19.1 MB 19.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/19.1 MB 16.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 8.7/19.1 MB 14.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━ 11.0/19.1 MB 13.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━ 13.6/19.1 MB 13.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 16.3/19.1 MB 13.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━

## 14. Save router metadata


In [20]:
metadata = {
    'project': 'VisionInspect',
    'model_type': 'domain_router',
    'architecture': MODEL_NAME,
    'classes': DOMAIN_TO_ID,
    'image_size': IMAGE_SIZE,
    'seed': SEED,
    'best_model': str(model.trainer.best),
    'onnx_model': str(onnx_path),
    'training_seconds': training_time,
    'test_accuracy': routing_accuracy,
    'wrong_specialist_rate': wrong_specialist_rate,
    'router_latency_ms': latency_ms,
}

metadata_path = ROUTER_ROOT / 'router_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(metadata_path)


datasets/domain_router/router_metadata.json


# Baseline complete

The router selects one of four specialist pipelines: DAGM, KolektorSDD2, Magnetic Tile, or MVTec AD.

Primary metrics: routing accuracy, macro F1, per-domain recall, confusion matrix, wrong-specialist rate, confidence/rejection trade-off, latency, and ONNX size.

Next: integrate the router with the specialist models in `05_integrated_inference_pipeline.ipynb`.
